# Water Level Data

### Code Summary:

- Detrending, normalizing, and shifting data to zero mean
- Subsampled data plots
- Tapered data plots
- Interactive plotly plots (Note: plotly plots work best for smaller code file sizes)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import scipy.stats as ss
from sklearn.preprocessing import MinMaxScaler
import scipy.fft
import pywt
import plotly.graph_objects as go
import plotly.tools as tls
from plotly.subplots import make_subplots
from scipy.signal.windows import hann
from scipy import interpolate
from scipy.ndimage import uniform_filter

In [ ]:
df1 = pd.read_csv("Toronto-daily-mean-water-1960-2024.csv", index_col="Obs_date")

# Check for null values in dataframe:
any_null = df1.isnull().any().any()
print(any_null) # Will show false if no null values

***
### Plotting the Time Series
#### Notes:
- Plot of the original water level data with no changes made
- Use the horizontal slider to zoom in/out of certain date ranges

In [ ]:
# Interactive plot of time series 
fig = go.Figure()
fig.add_trace(go.Scatter(x= list(df1.index), y= list(df1['SLEV(metres)']), mode='lines', name='Time Series'))
# Set title
fig.update_layout(title_text="Toronto Daily Mean Water Level (1960-2024)", width=1100, height=600, 
                  xaxis_title='Date', yaxis_title='Water Level (m)', title_x=0.5)
# Add range slider
fig.update_layout(xaxis=dict(rangeslider=dict(visible=True)))
fig.show()

print(ss.describe(df1['SLEV(metres)']))
print("mode:", ss.mode(df1['SLEV(metres)'])) 
print("standard deviation:", df1['SLEV(metres)'].std())
print("median:", df1["SLEV(metres)"].median())

***
### Normalizing, Detrending and Shifting Data
#### Notes:
- data range of time series was normalized using MinMaxScaler from sklearn
- data was linearly detrended using signal.detrend from scipy
- the mean of the data was shifted to approximately zero, as shown by the red dashed line on the plot

In [ ]:
# Linearly detrend data, normalize and shift to zero mean:
df1['Detrended'] = signal.detrend(df1['SLEV(metres)'], type='linear')

scaler = MinMaxScaler()
df1['Normalized'] = scaler.fit_transform(df1['Detrended'].values.reshape(-1, 1)).flatten()
print(f"Min-Max normalized range: [{df1['Normalized'].min():.3f}, {df1['Normalized'].max():.3f}]")

# Shift Mean:
df1['Normalized_Shifted'] = (df1['Normalized'] - np.mean(df1['Normalized']))
print(f"New range: [{df1['Normalized_Shifted'].min():.3f}, {df1['Normalized_Shifted'].max():.3f}]")
print(f"New mean: {df1['Normalized_Shifted'].mean():.6f}")

# Plot:
dshifted_norm_plot = go.Figure()
dshifted_norm_plot.add_trace(go.Scatter(x= list(df1.index), y= list(df1['Normalized_Shifted']), mode='lines', name='Time Series'))
dshifted_norm_plot.update_layout(title_text="Toronto Daily Mean Water Level (1960-2024) <br> (Detrended, Normalized, Shifted)",
                                 width=1100, height=600, xaxis_title='Date', yaxis_title='Normalized Water Level', title_x=0.5)
dshifted_norm_plot.update_layout(xaxis=dict(rangeslider=dict(visible=True)))

# Calculate the overall mean:
overall_mean = np.round(df1['Normalized_Shifted'].mean())
# Add a horizontal line for the overall mean
dshifted_norm_plot.add_shape(type="line", x0=df1.index.min(), y0=overall_mean, x1=df1.index.max(), y1=overall_mean, 
                    line=dict(color="Red", width=2, dash="dash"), name=f"Overall Mean ({overall_mean})", showlegend=True)

dshifted_norm_plot.show()

# Stats for plot
print(ss.describe(df1['Normalized_Shifted']))
print("median:", df1["Normalized_Shifted"].median())
print("mode:", ss.mode(df1["Normalized_Shifted"])) 
print("standard deviation:", df1["Normalized_Shifted"].std())

***
### Subsampling the Time Series
#### Notes:
- different step sizes were applied to the normalized, detrended and mean-shifted (DNS) data for subsampling
- click on the legend titles to select or hide certain time series
- use the horizontal slider to zoom in/out of certain date ranges

In [ ]:
# Subsampling time series:
# Number of data points in df1['Normalized_Shifted']: 23292
step1 = len(df1['Normalized_Shifted'])//100
step2 = len(df1['Normalized_Shifted'])//1000
step3 = len(df1['Normalized_Shifted'])//10000
step4 = 1000
step5 = 100
step6 = 10

# Plot:
dshifted_norm_plot = go.Figure()
dshifted_norm_plot.add_trace(go.Scatter(x= list(df1.index), y= list(df1['Normalized_Shifted']), mode='lines', name='Time Series (no step)'))
#dshifted_norm_plot.add_trace(go.Scatter(x= list(df1.index)[::step1], y= list(df1['Normalized_Shifted'][::step1]), mode='lines', 
#                                        name=f'Time Series (step={step1})'))
#dshifted_norm_plot.add_trace(go.Scatter(x= list(df1.index)[::step2], y= list(df1['Normalized_Shifted'][::step2]), mode='lines', 
#                                        name=f'Time Series (step={step2})'))
#dshifted_norm_plot.add_trace(go.Scatter(x= list(df1.index)[::step3], y= list(df1['Normalized_Shifted'][::step3]), mode='lines', 
#                                        name=f'Time Series (step={step3})'))
#dshifted_norm_plot.add_trace(go.Scatter(x= list(df1.index)[::step6], y= list(df1['Normalized_Shifted'][::step6]), mode='lines', 
#                                        name=f'Time Series (step={step6})'))
dshifted_norm_plot.add_trace(go.Scatter(x= list(df1.index)[::52], y= list(df1['Normalized_Shifted'][::52]), mode='lines', 
                                        name=f'Time Series (step={52})'))
#dshifted_norm_plot.add_trace(go.Scatter(x= list(df1.index)[::step5], y= list(df1['Normalized_Shifted'][::step5]), mode='lines', 
#                                        name=f'Time Series (step={step5})'))
dshifted_norm_plot.update_layout(title_text="Toronto Daily Mean Water Level (1960-2024) <br> Subsampling",
                                 width=1100, height=600, xaxis_title='Date', yaxis_title='Normalized Water Level', title_x=0.5)
dshifted_norm_plot.update_layout(xaxis=dict(rangeslider=dict(visible=True)))
dshifted_norm_plot.show()

In [ ]:
# stats for subsample step 52:
print("data step: 52")
print(ss.describe(df1['Normalized_Shifted'][::52]))
print("mode:", ss.mode(df1['Normalized_Shifted'][::52])) 
print("standard deviation:", df1['Normalized_Shifted'][::52].std())
print("median:", df1["Normalized_Shifted"][::52].median())

***
#### Updated Tapering Function - No Padding, Partial Taper
- Begin taper at 10% from the data edges
- A Hann window/raised cosine is used to taper the DNS time series

In [ ]:
# Updated tapering function
def apply_hann_window(data, column, taper_fraction = 0.1):
    """Taper edges of time series with Hann window
    taper_fraction = 0.1 will start taper at 10% from edges"""
    n= len(data)
    taper_length = int(n * taper_fraction)
    
    # Create a copy of the data
    windowed_data = data[column].copy()
    windowed_data = windowed_data.astype(float) # needed for FFT code to work properly
    
    # Apply Hann window to the left edge
    left_window = np.hanning(2 * taper_length)[:taper_length]
    windowed_data.iloc[:taper_length] *= left_window
    
    # Apply Hann window to the right edge
    right_window = np.hanning(2 * taper_length)[taper_length:]
    windowed_data.iloc[-taper_length:] *= right_window
    
    return windowed_data

***
### Combining Subsampling and Tapering
#### Notes:
- Taper applied with no padding to prevent discontinuities

In [ ]:
sub_step = 52 # change as needed

data = df1['Normalized_Shifted'].values
time_index = pd.to_datetime(df1.index)
# time delta (1 day for water level data)
time_delta = (time_index[1] - time_index[0])

windowed_data = apply_hann_window(df1, 'Normalized_Shifted') # uses updated tapering function
tapered_sub = windowed_data[::sub_step]

# Plot:
sub_t = go.Figure()
sub_t.add_trace(go.Scatter(x= list(time_index), y= list(df1['Normalized_Shifted']), mode='lines', name='Full Time Series')) 
sub_t.add_trace(go.Scatter(x= list(time_index)[::sub_step], y= list(tapered_sub), mode='lines', name=f'Tapered Time Series (step={sub_step})'))
sub_t.update_layout(title_text="Tapered and Subsampled Toronto Daily Mean Water Level (1960-2024)", width=1100, height=600,
    xaxis_title='Date', yaxis_title='Normalized Water Level', title_x=0.5)
sub_t.update_layout(xaxis=dict(rangeslider=dict(visible=True)))
sub_t.show()

# Summary Stats for tapered/subsampled series:
print("Summary Stats:")
print(ss.describe(tapered_sub))
print("mode:", ss.mode(tapered_sub)) 
print("standard deviation:", tapered_sub.std())
print("median:", np.median(tapered_sub))

In [ ]:
# see next file for adding sine waves, FFT plots and scalograms